In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [100]:
# df = pd.read_csv('data/pre-thin-data.csv')
# # df = pd.read_csv('data/clean-thinning-data-2.csv')
# # df = pd.read_csv('data/clean-thinning-data-3.csv')

# df['status'] = df['pre_HT'].apply(lambda x: 'Alive' if pd.notnull(x) and x > 0 else 'Dead')

def load_stand_csv(path: str) -> pd.DataFrame:
    df_ = pd.read_csv(path)


    # Make/normalize 'status'
    if "status" not in df_.columns:
        if "pre_HT" in df_.columns:
            df_["status"] = df_["pre_HT"].apply(lambda x: "Alive" if pd.notnull(x) and x > 0 else "Dead")
        else:
            df_["status"] = "Alive"

    return df_


In [104]:
df = load_stand_csv(stand_dd.value)

# Find the best start row 
df_best3, best_start3, ranked3 = compute_best3(df)

# Build the three 3-row baselines (start_row = 1, 2, 3)
df_k3_by_start = {
    s: k_row_thinning(df, 3, start_row=s, row_col='Row', status_col='status')
    for s in (1, 2, 3)
}

### Adjusting Thinning Strategies on Section

In [108]:
# Initial thinning and secondary thinning libs
from initial_thinning_lib import compute_best3, k_row_thinning, plot_thinning_map
from secondary_thinning_lib import (
    table_final_vs_initial,
    table_final_vs_after_first,
    thin_from_below_adjacent_simple,         # TFB 
    thin_from_above_neighbors,               # TFA-1 
    thin_from_above_phase2_anchor_immediate5, # TFA-2 
    anchor_release_table_immediate5
)


def quadrant_bounds(df, row_col='Row', x_col='Tree'):
    rows = df[row_col].to_numpy()
    xs   = df[x_col].to_numpy()
    r_med = int(np.median(rows))
    x_med = int(np.median(xs))

    return {
        'NW': {'row_lo': df[row_col].min(), 'row_hi': r_med,
               'x_lo': df[x_col].min(),     'x_hi': x_med},
        'NE': {'row_lo': df[row_col].min(), 'row_hi': r_med,
               'x_lo': x_med+1,             'x_hi': df[x_col].max()},
        'SW': {'row_lo': r_med+1,           'row_hi': df[row_col].max(),
               'x_lo': df[x_col].min(),     'x_hi': x_med},
        'SE': {'row_lo': r_med+1,           'row_hi': df[row_col].max(),
               'x_lo': x_med+1,             'x_hi': df[x_col].max()}
    }

def mask_in_bounds(df, b, row_col='Row', x_col='Tree'):
    return (df[row_col].between(b['row_lo'], b['row_hi'])) & (df[x_col].between(b['x_lo'], b['x_hi']))

# --------------------------------------------------------------------
# run each strategy only inside a section
# --------------------------------------------------------------------

def tfb_section_picks(df_best3, bounds, *, fraction=1/3,
                      metric='pre_DBH', row_col='Row', x_col='Tree', status_col='status',
                      allow_distance2=False):
    """
    Thin-from-below (sectional) with Hamilton apportionment.

    • Candidates = Alive & Keep on side-rows (rows adjacent to 3-row corridors) AND inside the section.
    • Section target = round(total_eligible_in_section * fraction).
    • Allocate per-row with Hamilton (largest remainder) up to capacity.
    • If shortfall remains, take additional smallest-DBH trees from the leftover side-row candidates
      (still inside the section) until the section hits the exact target.
    • If capacity of side-rows is still insufficient and allow_distance2=True, expand to rows at distance 2
      (rc±2).
    """
    d = df_best3.copy()
    alive = d[status_col].eq('Alive')

    # 3-row corridor rows
    corridor_mask = alive & d['thin_decision'].eq('Thin')
    corridor_rows = np.sort(d.loc[corridor_mask, row_col].unique())
    if len(corridor_rows) == 0:
        return pd.Index([])

    in_sec = mask_in_bounds(d, bounds, row_col=row_col, x_col=x_col)

    # Side rows (rc±1), inside section
    side_rows_all, all_rows = [], np.sort(d[row_col].unique())
    corridors = set(corridor_rows)
    for rc in corridor_rows:
        if (rc - 1) in all_rows and (rc - 1) not in corridors: side_rows_all.append(rc - 1)
        if (rc + 1) in all_rows and (rc + 1) not in corridors: side_rows_all.append(rc + 1)
    side_rows_all = sorted(set(side_rows_all))
    if not side_rows_all:
        return pd.Index([])

    # Eligible per side row
    per_row_elig = {}
    n_per_row = {}
    for r in side_rows_all:
        m = alive & d['thin_decision'].eq('Keep') & d[row_col].eq(r) & in_sec
        idx = d.index[m]
        per_row_elig[r] = idx
        n_per_row[r]    = int(len(idx))

    total_elig = int(sum(n_per_row.values()))
    if total_elig == 0:
        return pd.Index([])

    # Section target
    target = int(np.round(total_elig * float(fraction)))
    if target <= 0:
        return pd.Index([])

    # Hamilton apportionment
    rows_list = [r for r in side_rows_all if n_per_row[r] > 0]
    q_ideal = {r: n_per_row[r] * float(fraction) for r in rows_list}
    q_floor = {r: int(np.floor(q_ideal[r]))      for r in rows_list}
    q_frac  = {r: q_ideal[r] - q_floor[r]        for r in rows_list}

    quota = {r: min(q_floor[r], n_per_row[r]) for r in rows_list}
    assigned = int(sum(quota.values()))
    remaining = target - assigned

    if remaining > 0:
        prio = sorted(rows_list, key=lambda r: (q_frac[r], n_per_row[r], r), reverse=True)
        i = 0
        while remaining > 0 and any(quota[r] < n_per_row[r] for r in prio):
            r = prio[i % len(prio)]
            if quota[r] < n_per_row[r]:
                quota[r] += 1
                remaining -= 1
            i += 1

    picks = []
    leftovers = []
    for r in rows_list:
        idx = per_row_elig[r]
        if len(idx) == 0:
            continue
        sub = d.loc[idx, [metric]].sort_values(by=metric, ascending=True)
        k_rm = int(min(quota[r], len(sub)))
        if k_rm > 0:
            chosen = sub.index[:k_rm]
            picks.append(chosen)
            rest = sub.index[k_rm:]
        else:
            chosen = []
            rest = sub.index
        if len(rest) > 0:
            leftovers.append(rest)

    if picks:
        picks = pd.Index(np.concatenate([ix.values for ix in picks])).unique()
    else:
        picks = pd.Index([])

    # Local backfill from the remaining side-row candidates
    shortfall = target - len(picks)
    if shortfall > 0 and len(leftovers) > 0:
        pool = pd.Index(np.concatenate([ix.values for ix in leftovers]))
        if len(pool) > 0:
            pool_sorted = d.loc[pool, [metric]].sort_values(by=metric, ascending=True)
            add = pool_sorted.index[:min(shortfall, len(pool_sorted))]
            picks = picks.union(add)
            shortfall = target - len(picks)

    if shortfall > 0 and allow_distance2:
        rows_d2 = []
        for rc in corridor_rows:
            for r2 in (rc - 2, rc + 2):
                if (r2 in all_rows) and (r2 not in corridors):
                    rows_d2.append(r2)
        rows_d2 = sorted(set(rows_d2))

        d2_pools = []
        for r in rows_d2:
            m = alive & d['thin_decision'].eq('Keep') & d[row_col].eq(r) & in_sec
            idx = d.index[m]
            if len(idx) > 0:
                d2_pools.append(idx)
        if len(d2_pools) > 0:
            pool2 = pd.Index(np.concatenate([ix.values for ix in d2_pools]))
            pool2 = pool2.difference(picks)
            if len(pool2) > 0:
                pool2_sorted = d.loc[pool2, [metric]].sort_values(by=metric, ascending=True)
                add2 = pool2_sorted.index[:min(shortfall, len(pool2_sorted))]
                picks = picks.union(add2)
                shortfall = target - len(picks)

    return picks


def tfa1_section_picks(df_best3, bounds, *,
                       removal_fraction=1/3, metric='pre_DBH',
                       row_col='Row', x_col='Tree', status_col='status',
                       thin_col='thin_decision', keep_val='Keep', thin_val='Thin',
                       anchor_fraction=0.10, min_anchors=1, radius=None, combine='max',
                       do_backfill=True):
    """
    Thin-from-above-1 -- sectional with Hamilton apportionment
    Target base = sum over side-rows of eligible trees inside the section.
    """

    d0 = df_best3.copy()
    in_sec = mask_in_bounds(d0, bounds, row_col=row_col, x_col=x_col)
    alive  = d0[status_col].eq('Alive')

    # corridor rows are global (from the 3-row pass)
    corridor_rows = np.sort(d0.loc[alive & d0[thin_col].eq(thin_val), row_col].unique())
    if len(corridor_rows) == 0:
        return pd.Index([])

    # side rows = rc ± 1, excluding corridors
    side_rows_all, all_rows = [], np.sort(d0[row_col].unique())
    corridors = set(corridor_rows)
    for rc in corridor_rows:
        if (rc - 1) in all_rows and (rc - 1) not in corridors: side_rows_all.append(rc - 1)
        if (rc + 1) in all_rows and (rc + 1) not in corridors: side_rows_all.append(rc + 1)
    side_rows_all = sorted(set(side_rows_all))
    if not side_rows_all:
        return pd.Index([])

    # Per-row eligible (Alive & Keep & inside section on that row)
    per_row_eligible_idx = {}
    per_row_anchors      = {}
    per_row_pool         = {}   # candidates that can be cut (non-anchors with score)
    per_row_scores       = {}   # score series for candidates
    per_row_all_scores   = {}   # keep for backfill

    for r in side_rows_all:
        elig = alive & d0[thin_col].eq(keep_val) & d0[row_col].eq(r) & in_sec
        idx  = d0.index[elig]
        per_row_eligible_idx[r] = idx

        if len(idx) == 0:
            per_row_anchors[r]    = pd.Index([])
            per_row_pool[r]       = pd.Index([])
            per_row_scores[r]     = pd.Series(dtype=float)
            per_row_all_scores[r] = pd.Series(dtype=float)
            continue

        # anchors on this row by DBH
        sub = d0.loc[idx, [metric, x_col]]
        k_anchor = max(min_anchors, int(np.ceil(len(sub) * float(anchor_fraction))))
        k_anchor = min(k_anchor, len(sub))
        anchors_idx = sub.nlargest(k_anchor, metric).index
        per_row_anchors[r] = anchors_idx

        # score every non-anchor by influence of row anchors 
        a_pos    = d0.loc[anchors_idx, x_col].astype(float).to_numpy()
        a_weight = d0.loc[anchors_idx, metric].astype(float).to_numpy()

        row_scores = {}
        for i, row_i in sub.iterrows():
            if i in anchors_idx:
                continue
            xi = float(row_i[x_col])
            if radius is None:
                dx  = np.abs(a_pos - xi)
                inf = a_weight / (dx + 1.0)
            else:
                dx  = np.abs(a_pos - xi)
                m   = dx <= float(radius)
                if not m.any():
                    continue
                inf = a_weight[m] / (dx[m] + 1.0)

            s_i = np.max(inf) if combine == 'max' else np.sum(inf)
            row_scores[i] = float(s_i)

        # pool = non-anchors that actually got a score
        if len(row_scores) == 0:
            per_row_pool[r]       = pd.Index([])
            per_row_scores[r]     = pd.Series(dtype=float)
            per_row_all_scores[r] = pd.Series(dtype=float)
        else:
            s = pd.Series(row_scores)
            per_row_pool[r]       = s.index
            per_row_scores[r]     = s
            per_row_all_scores[r] = s  # keep copy for backfill

    # ---------- Hamilton on ELIGIBLE counts ----------
    eligible_counts = {r: int(len(per_row_eligible_idx[r])) for r in side_rows_all}
    total_eligible  = int(sum(eligible_counts.values()))
    if total_eligible == 0:
        return pd.Index([])

    section_target = int(np.round(total_eligible * float(removal_fraction)))

    # ideal, floors, remainders
    q_ideal = {r: eligible_counts[r] * float(removal_fraction) for r in side_rows_all}
    q_floor = {r: int(np.floor(q_ideal[r])) for r in side_rows_all}
    q_frac  = {r: q_ideal[r] - q_floor[r]   for r in side_rows_all}

    # start with floors, but cap by each row's *pool* capacity
    pool_sizes = {r: int(len(per_row_pool[r])) for r in side_rows_all}
    quota      = {r: min(q_floor[r], pool_sizes[r]) for r in side_rows_all}
    assigned   = int(sum(quota.values()))
    remaining  = section_target - assigned

    # distribute remainders by largest fractional part, respecting capacity
    if remaining > 0:
        order = sorted(side_rows_all, key=lambda r: (q_frac[r], eligible_counts[r], r), reverse=True)
        i = 0
        spins = 0
        while remaining > 0 and spins < 10_000:
            r = order[i % len(order)]
            if quota[r] < pool_sizes[r]:
                quota[r] += 1
                remaining -= 1
            i += 1
            spins += 1
            # if every row is saturated, break
            if all(quota[x] >= pool_sizes[x] for x in side_rows_all):
                break

    # ---------- Per-row take up to the quota ----------
    picks = []
    taken = set()
    for r in side_rows_all:
        k = quota.get(r, 0)
        if k <= 0 or pool_sizes[r] == 0:
            continue
        s = per_row_scores[r].sort_values(ascending=False)
        choose = [i for i in s.index if i not in taken][:k]
        picks.extend(choose)
        taken.update(choose)

    if do_backfill:
        short = section_target - len(picks)
        if short > 0:
            leftovers = []
            for r in side_rows_all:
                s = per_row_all_scores[r].drop(index=list(taken), errors='ignore')
                leftovers.append(s)
            if leftovers:
                glob = pd.concat(leftovers).sort_values(ascending=False)
                add  = list(glob.index[:short])
                picks.extend(add)
                taken.update(add)

    return pd.Index(picks).unique()



def _euclid2d(a_rows, a_trees, b_rows, b_trees, row_scale=1.0, tree_scale=1.0):
    dr = (a_rows[:,None]-b_rows[None,:])*row_scale
    dt = (a_trees[:,None]-b_trees[None,:])*tree_scale
    return np.sqrt(dr*dr + dt*dt)

def tfa2_immediatek_section_picks(
    df_best3, bounds, *,
    dbh_col='pre_DBH', row_col='Row', x_col='Tree',
    status_col='status', thin_col='thin_decision',
    keep_val='Keep', thin_val='Thin',
    anchor_frac=0.10,            # share of baseline used as anchors 
    k_neighbors=5,               # initial k for immediate windows
    removed_frac=1/3,
    row_scale=1.0, tree_scale=1.0,
    k_step=2,                    # expand k by this many when short
    k_max=None,                  # cap for k
    do_score_backfill=True       # after expand-k, do score-sum fill
):
    d0 = df_best3.copy()
    in_sec = mask_in_bounds(d0, bounds, row_col=row_col, x_col=x_col)

    # Baseline
    base_mask = in_sec & d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    B_idx = d0.index[base_mask]
    if len(B_idx) == 0:
        return pd.Index([])

    B = d0.loc[B_idx, [dbh_col, row_col, x_col]]
    n_base = len(B)

    # Target for this section
    target = int(np.round(n_base * float(removed_frac)))
    if target <= 0:
        return pd.Index([])

    # Anchors (top DBH within the section)
    n_anchors = max(1, int(np.ceil(n_base * float(anchor_frac))))
    anchors = B.nlargest(n_anchors, dbh_col)
    anchor_ids = anchors.index
    anchor_set = set(anchor_ids)

    # Arrays for distances
    rows = B[row_col].to_numpy(float)
    cols = B[x_col].to_numpy(float)
    ids  = np.array(list(B.index))
    id2pos = {ids[i]: i for i in range(len(ids))}

    ar = anchors[row_col].to_numpy(float)
    ac = anchors[x_col].to_numpy(float)
    aw = anchors[dbh_col].to_numpy(float)

    def build_windows(k_now: int) -> dict[int, np.ndarray]:
        """Nearest-k windows for each anchor."""
        win = {}
        for a in anchor_ids:
            ai = id2pos[a]
            D = _euclid2d(np.array([rows[ai]]), np.array([cols[ai]]),
                          rows, cols, row_scale=row_scale, tree_scale=tree_scale).ravel()
            D[ai] = np.inf
            k_eff = min(int(k_now), len(ids) - 1)
            if k_eff <= 0:
                win[a] = np.array([], dtype=ids.dtype)
                continue
            sel = np.argpartition(D, k_eff - 1)[:k_eff]
            sel = sel[np.argsort(D[sel])]              # nearest-first
            win[a] = ids[sel]
        return win

    # ---- Phase A: immediate-k 
    picks = []
    budget = target
    k_curr = min(int(k_neighbors), len(ids) - 1)
    windows = build_windows(k_curr)

    for a in anchors.index:
        if budget <= 0:
            break
        for nid in windows[a]:
            if budget <= 0:
                break
            if nid in anchor_set:
                continue
            if d0.at[nid, thin_col] != keep_val: 
                continue
            if nid in picks:
                continue
            picks.append(nid)
            budget -= 1

    # ---- Backfill A: expand-k in steps until target or capacity reached
    if budget > 0:
        k_cap = (len(ids) - 1) if k_max is None else min(int(k_max), len(ids) - 1)
        while budget > 0 and k_curr < k_cap:
            k_curr = min(k_curr + int(k_step), k_cap)
            windows = build_windows(k_curr)
            added_any = False
            for a in anchors.index:
                if budget <= 0:
                    break
                for nid in windows[a]:
                    if budget <= 0:
                        break
                    if nid in anchor_set:
                        continue
                    if d0.at[nid, thin_col] != keep_val:
                        continue
                    if nid in picks:
                        continue
                    picks.append(nid)
                    budget -= 1
                    added_any = True
            if not added_any:
                break  

    # ---- Backfill B: anchor-proximity score (sum of aw/(d+1))
    if budget > 0 and do_score_backfill:
        # remaining non-anchors in the section baseline
        remaining_mask = ~B.index.isin(picks) & ~B.index.isin(anchor_set)
        if remaining_mask.any():
            R = B.loc[remaining_mask]
            rr = R[row_col].to_numpy(float)
            rc = R[x_col].to_numpy(float)
            rid = R.index.to_numpy()

            D = _euclid2d(rr, rc, ar, ac, row_scale=row_scale, tree_scale=tree_scale)
            with np.errstate(divide='ignore', invalid='ignore'):
                contrib = aw / (D + 1.0)     # broadcast
            contrib[np.isinf(contrib)] = 0.0
            score = np.nan_to_num(contrib, nan=0.0).sum(axis=1)

            order = np.argsort(-score)
            take  = rid[order[:budget]]
            picks.extend(list(take))
            budget = 0 

    return pd.Index(picks).unique()



def q4_immediatek_prefer_q12_section_picks(
    df_best3, bounds, *,
    dbh_col='pre_DBH', row_col='Row', x_col='Tree',
    status_col='status', thin_col='thin_decision',
    keep_val='Keep', thin_val='Thin',
    q4_fraction=0.25,          # fraction of section baseline used as anchors
    k_neighbors=5,             # initial k for immediate window
    removed_frac=1/3,          # target removal fraction
    row_scale=1.0, tree_scale=1.0,
    target_rounding='ceil',    # 'ceil' ensures each section rounds UP to meet quota
    k_step=2,                  # expand k by this step if short
    k_max=None,                # max k used during expansion
    distance_cap=None,         # cap anchor influence distance in score backfill
    allow_loose_backfill=True, # final fill with smallest Q1/Q2 if still short
    verbose=False
) -> pd.Index:
    
    """
    Sectional Q4 immediate-k with Q1->Q2 preference
    """
    d0 = df_best3.copy()

    in_sec = (
        d0[row_col].between(bounds['row_lo'], bounds['row_hi'])
        & d0[x_col].between(bounds['x_lo'], bounds['x_hi'])
    )
    base_mask = in_sec & d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    B_idx = d0.index[base_mask]
    if len(B_idx) == 0:
        return pd.Index([])

    B = d0.loc[B_idx, [dbh_col, row_col, x_col]]
    n_base = len(B)

    if target_rounding == 'floor':
        target = int(np.floor(n_base * float(removed_frac)))
    elif target_rounding == 'round':
        target = int(np.round(n_base * float(removed_frac)))
    else:  # 'ceil' (default)
        target = int(np.ceil(n_base * float(removed_frac)))
    target = max(0, min(target, n_base))
    if target == 0:
        return pd.Index([])

    x = B[dbh_col].astype(float)
    q1_thr = float(x.quantile(0.25))
    q2_thr = float(x.quantile(0.50))

    n_anchors = max(1, int(np.ceil(n_base * float(q4_fraction))))
    anchors = B.nlargest(n_anchors, dbh_col)
    anchor_ids = anchors.index
    anchor_set = set(anchor_ids)

    # Arrays for distances
    rows = B[row_col].to_numpy(float)
    cols = B[x_col].to_numpy(float)
    ids  = np.array(list(B.index))
    id2pos = {ids[i]: i for i in range(len(ids))}

    # dists[a] = distances from anchor 'a' to ALL B points (anchors included)
    def _euclid2d(a_rows, a_cols, b_rows, b_cols, rs=1.0, cs=1.0):
        dr = (a_rows[:, None] - b_rows[None, :]) * rs
        dc = (a_cols[:, None] - b_cols[None, :]) * cs
        return np.sqrt(dr * dr + dc * dc)

    a_rows = anchors[row_col].to_numpy(float)[:, None]
    a_cols = anchors[x_col].to_numpy(float)[:, None]
    D_all  = _euclid2d(a_rows.ravel(), a_cols.ravel(), rows, cols,
                       rs=row_scale, cs=tree_scale)  # shape: [n_anchors, n_base]

    # For each anchor, an ordered list of neighbor IDs excluding itself
    near_sorted = []
    for j, a in enumerate(anchors.index):
        d = D_all[j].copy()
        ai = id2pos[a]
        d[ai] = np.inf
        order = np.argsort(d)        
        near_sorted.append(ids[order])

    B_dbh = B[dbh_col].astype(float)
    is_q1B = B_dbh <= q1_thr
    is_q2B = (B_dbh > q1_thr) & (B_dbh <= q2_thr)

    picks = []
    picked = set()
    budget = target

    def _take_from_windows(k):
        """Take Q1 then Q2 from each anchor's first k neighbors, nearest-first."""
        nonlocal budget
        if budget <= 0:
            return 0
        taken_here = 0
        for arr in near_sorted:
            if budget <= 0:
                break
            cand = arr[:min(k, len(arr))]
            # keep order; split into Q1 then Q2
            q1_ids = [int(i) for i in cand if is_q1B.get(i, False)]
            q2_ids = [int(i) for i in cand if is_q2B.get(i, False)]
            tiered = q1_ids + q2_ids
            for nid in tiered:
                if budget <= 0:
                    break
                if nid in picked:
                    continue
                if nid in anchor_set:
                    continue
    
                if d0.at[nid, thin_col] != keep_val:
                    continue
                picked.add(nid); picks.append(nid); budget -= 1; taken_here += 1
        return taken_here


    is_q1B = is_q1B.to_dict()
    is_q2B = is_q2B.to_dict()

    # -------------------
    # Phase A: strict k
    # -------------------
    _take_from_windows(int(k_neighbors))

    # -------------------
    # Phase B: expand k
    # -------------------
    if budget > 0 and k_step and k_step > 0:
        if k_max is None:
            k_max = max(int(k_neighbors), 9)
        k = int(k_neighbors) + int(k_step)
        while budget > 0 and k <= int(k_max):
            got = _take_from_windows(k)
            if got == 0:
                break
            k += int(k_step)

    # -------------------
    # Phase C: proximity score backfill
    # -------------------
    if budget > 0:
        remain_mask = B.index.difference(pd.Index(picks))
        if len(remain_mask) > 0:
            C = B.loc[remain_mask]
            c_rows = C[row_col].to_numpy(float)
            c_cols = C[x_col].to_numpy(float)
            # distances candidate -> each anchor
            D = _euclid2d(c_rows, c_cols, anchors[row_col].to_numpy(float),
                          anchors[x_col].to_numpy(float), rs=row_scale, cs=tree_scale)
            if distance_cap is not None:
                D = np.where(D <= float(distance_cap), D, np.inf)
            with np.errstate(divide='ignore', invalid='ignore'):
                contrib = anchors[dbh_col].to_numpy(float)[None, :] / (D + 1.0)
            contrib[np.isinf(contrib)] = 0.0
            score = np.nan_to_num(contrib, nan=0.0).sum(axis=1)

            C_dbh = C[dbh_col].astype(float)
            mask_q12 = (C_dbh <= q2_thr)  # only Q1/Q2
            if mask_q12.any():
                C_q12 = C.loc[mask_q12]
                score_q12 = pd.Series(score, index=C.index).loc[C_q12.index]
                order = score_q12.sort_values(ascending=False).index.tolist()
                for nid in order:
                    if budget <= 0:
                        break
                    if nid in picked or nid in anchor_set:
                        continue
                    if d0.at[nid, thin_col] != keep_val:
                        continue
                    picked.add(int(nid)); picks.append(int(nid)); budget -= 1

    # -------------------
    # Phase D: smallest Q1/Q2
    # -------------------
    if budget > 0 and allow_loose_backfill:
        remain_mask = B.index.difference(pd.Index(picks))
        C = B.loc[remain_mask]
        C_q12 = C.loc[C[dbh_col].astype(float) <= q2_thr]
        if len(C_q12) > 0:
            need = min(budget, len(C_q12))
            tail = C_q12.nsmallest(need, dbh_col).index.tolist()
            for nid in tail:
                if budget <= 0:
                    break
                if nid in picked or nid in anchor_set:
                    continue
                if d0.at[nid, thin_col] != keep_val:
                    continue
                picked.add(int(nid)); picks.append(int(nid)); budget -= 1

    if verbose:
        print(f"[A2 fixed] target={target}, picked={len(picks)}, short_by={max(0, target-len(picks))}")

    return pd.Index(picks).unique()



def q12_weighted_by_q4_section_picks(df_best3, bounds, *,
                                     dbh_col='pre_DBH', row_col='Row', x_col='Tree',
                                     status_col='status', thin_col='thin_decision',
                                     keep_val='Keep', thin_val='Thin',
                                     removed_frac=1/3,
                                     row_scale=1.0, tree_scale=1.0,
                                     w_q1=1.0, w_q2=0.6, w_q3=0.15, w_q4=0.0):
    d0 = df_best3.copy()
    in_sec = mask_in_bounds(d0, bounds, row_col=row_col, x_col=x_col)
    baseline_mask = in_sec & d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    B_idx = d0.index[baseline_mask]
    if len(B_idx) == 0:
        return pd.Index([])
    B = d0.loc[B_idx, [dbh_col, row_col, x_col]]
    target = int(round(len(B_idx)*removed_frac))
    if target <= 0:
        return pd.Index([])

    # Quartiles within the Section
    x = B[dbh_col].astype(float)
    q1 = x.quantile(0.25); q2 = x.quantile(0.50); q3 = x.quantile(0.75)

    # anchor set = local Q4
    anchors = B.loc[x >= q3]
    if len(anchors) == 0:
        
        return B.nsmallest(target, dbh_col).index
        
    rows = B[row_col].to_numpy(float); cols = B[x_col].to_numpy(float); ids = np.array(list(B_idx))
    id2pos = {ids[i]: i for i in range(len(ids))}
    ar = anchors[row_col].to_numpy(float); ac = anchors[x_col].to_numpy(float); aw = anchors[dbh_col].to_numpy(float)

    
    D = _euclid2d(rows, cols, ar, ac, row_scale=row_scale, tree_scale=tree_scale)  
    with np.errstate(divide='ignore', invalid='ignore'):
        influence = (aw / (D + 1.0))  
    infl_score = np.nan_to_num(influence, nan=0.0, posinf=0.0, neginf=0.0).sum(axis=1)
    
    w = np.where(x <= q1, w_q1,
         np.where(x <= q2, w_q2,
         np.where(x <= q3, w_q3, w_q4)))
    
    score = w * infl_score

    cut_order = pd.Series(score, index=B.index).sort_values(ascending=False)
    picks = list(cut_order.index[:target])

    if len(picks) < target:
        need = target - len(picks)
        backfill = B.loc[~B.index.isin(picks)].nsmallest(need, dbh_col).index.tolist()
        picks += backfill

    return pd.Index(picks).unique()

def run_sectional_strategy(df_best3, strategy_key, *,
                           removed_frac=1/3, anchor_frac=0.10, k_neighbors=5,
                           row_scale=1.0, tree_scale=1.0):
    d0 = df_best3.copy()
    QB = quadrant_bounds(d0)  
    picks_all = []

    for qname, b in QB.items():
        if strategy_key == 'tfb':
            picks = tfb_section_picks(d0, b, fraction=removed_frac)
        elif strategy_key == 'tfa1':
            picks = tfa1_section_picks(d0, b, removal_fraction=removed_frac, anchor_fraction=anchor_frac)
        elif strategy_key == 'tfa2':
            picks = tfa2_immediatek_section_picks(
                d0, b,
                removed_frac=removed_frac,
                anchor_frac=anchor_frac,
                k_neighbors=k_neighbors,
                row_scale=row_scale,
                tree_scale=tree_scale,
                k_step=2,          
                k_max=7,        
                do_score_backfill=False
            )

        elif strategy_key == 'q4_immediate_k':
            picks = q4_immediatek_prefer_q12_section_picks(
                d0, b,
                q4_fraction=anchor_frac,          
                k_neighbors=k_neighbors,          
                removed_frac=removed_frac,        # section budget
                row_scale=row_scale, tree_scale=tree_scale,
                target_rounding='ceil',           # helps hit quota
                k_step=2, k_max=max(k_neighbors, 9),
                distance_cap=1.0,                 
                allow_loose_backfill=False         # final fill with smallest Q1/Q2
            )

        elif strategy_key == 'q12_weighted_q4':
            picks = q12_weighted_by_q4_section_picks(d0, b, removed_frac=removed_frac, row_scale=row_scale, tree_scale=tree_scale)
        else:
            raise ValueError("Unknown strategy key")
        picks_all.append(picks)

    all_picks = pd.Index(pd.unique(np.concatenate([p.values for p in picks_all if len(p)>0]))) if any(len(p)>0 for p in picks_all) else pd.Index([])
    df_out = df_best3.copy()
    if len(all_picks) > 0:
        df_out.loc[all_picks, 'thin_decision'] = 'Thin'
    return df_out, QB


### UI

In [107]:
pd.set_option('display.max_columns', None)   
pd.set_option('display.width', 0)            
pd.set_option('display.max_colwidth', None) 


def plot_thinning_map_with_sections(df_thinned, *,
                                    row_col='Row', x_col='Tree', status_col='status',
                                    title='Spatial map', r_med=None, x_med=None,
                                    annotate_labels=True):
    d = df_thinned
    alive  = d[d[status_col] == 'Alive']
    kept   = alive[alive['thin_decision'] == 'Keep']
    thin   = alive[alive['thin_decision'] == 'Thin']
    dead   = d[d[status_col] == 'Dead'] if 'Dead' in d[status_col].unique() else d.iloc[0:0]

    if r_med is None: r_med = int(np.median(d[row_col].to_numpy()))
    if x_med is None: x_med = int(np.median(d[x_col].to_numpy()))

    fig, ax = plt.subplots(figsize=(8, 8))
    if len(dead) > 0:
        ax.scatter(dead[x_col], dead[row_col], s=18, c='gray', alpha=0.5,
                   label=f'Dead (n={len(dead)})', edgecolors='none')
    if len(kept) > 0:
        ax.scatter(kept[x_col], kept[row_col], s=21, c='green',
                   label=f'Kept (n={len(kept)})', edgecolors='k', linewidths=0.25)
    if len(thin) > 0:
        ax.scatter(thin[x_col], thin[row_col], s=18, c='red', alpha=0.3,
                   label=f'Thinned (n={len(thin)})')

    ax.axvline(x=x_med + 0.5, linestyle='--', linewidth=1.2, alpha=0.8, color='black')
    ax.axhline(y=r_med + 0.5, linestyle='--', linewidth=1.2, alpha=0.8, color='black')

    if annotate_labels:
        xmin, xmax = d[x_col].min(), d[x_col].max()
        ymin, ymax = d[row_col].min(), d[row_col].max()
        ax.text(xmin + (x_med - xmin) / 2, ymin + (r_med - ymin) / 2, 'section-1',
                ha='center', va='center', fontsize=10, alpha=0.7)
        ax.text(x_med + 1 + (xmax - (x_med + 1)) / 2, ymin + (r_med - ymin) / 2, 'section-2',
                ha='center', va='center', fontsize=10, alpha=0.7)
        ax.text(xmin + (x_med - xmin) / 2, r_med + 1 + (ymax - (r_med + 1)) / 2, 'section-3',
                ha='center', va='center', fontsize=10, alpha=0.7)
        ax.text(x_med + 1 + (xmax - (x_med + 1)) / 2, r_med + 1 + (ymax - (r_med + 1)) / 2, 'section-4',
                ha='center', va='center', fontsize=10, alpha=0.7)

    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Tree (column)')
    ax.set_ylabel('Row')
    ax.set_title(title)
    ax.grid(True, linewidth=0.5, alpha=0.5)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0.)
    fig.subplots_adjust(right=0.78)
    plt.tight_layout()
    return fig, ax, r_med, x_med

# --- Stand dropdown
stand_dd = widgets.Dropdown(
    options=[
        ("Stand A - Dillwyn",      "data/pre-thin-data.csv"),
        ("Stand B",  "data/clean-thinning-data-2.csv"),
        ("Stand C",  "data/clean-thinning-data-3.csv"),
    ],
    value="data/pre-thin-data.csv",
    description="Stand:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="360px"),
)


# 3-row thinning option selection
k3_start_dd = widgets.Dropdown(
    options=[(f"3-row start = {s}" + (" (best)" if s == best_start3 else ""), s) for s in (1, 2, 3)],
    value=best_start3,
    description='Baseline:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='260px')
)

# Strategy dropdown
strategy_dd = widgets.Dropdown(
    options=[
        ('Thin from below', 'tfb'),
        ('Thin from above-1', 'tfa1'),
        ('Thin from above-2', 'tfa2'),
        ('Q4 immediate-k', 'q4_immediate_k'),
        ('Q1/Q2 weighted by Q4', 'q12_weighted_q4'),
    ],
    value='tfa2', description='Strategy:',
    style={'description_width':'110px'},
    layout=widgets.Layout(width='420px')
)
removed_frac_sl = widgets.FloatSlider(value=1/3, min=0.05, max=0.6, step=0.01,
                                      description='Remove frac:', readout_format='.2f',
                                      style={'description_width':'110px'},
                                      layout=widgets.Layout(width='420px'))
anchor_frac_sl  = widgets.FloatSlider(value=0.10, min=0.05, max=0.40, step=0.01,
                                      description='Anchor frac:', readout_format='.2f',
                                      style={'description_width':'110px'},
                                      layout=widgets.Layout(width='420px'))
k_neighbors_sl  = widgets.IntSlider(value=5, min=1, max=15, step=1,
                                    description='k neighbors:',
                                    style={'description_width':'110px'},
                                    layout=widgets.Layout(width='420px'))

ctrls = widgets.VBox([stand_dd, k3_start_dd,strategy_dd, removed_frac_sl, anchor_frac_sl, k_neighbors_sl])

out_full    = widgets.Output()
out_release = widgets.Output()
out_s1      = widgets.Output()
out_s2      = widgets.Output()
out_s3      = widgets.Output()
out_s4      = widgets.Output()
out_map     = widgets.Output()

SECTION_ORDER = [
    ('section-1', 'NW'),
    ('section-2', 'NE'),
    ('section-3', 'SW'),
    ('section-4', 'SE'),
]

def _show_section_table(df_initial, df_after_first, df_final, bounds, section_name, title_prefix):
    
    sec_mask = mask_in_bounds(df_after_first, bounds)
    if sec_mask.sum() == 0:
        display(HTML(f"<h4>{title_prefix} — {section_name}</h4>"))
        display(pd.DataFrame([{'Note': 'No trees in section'}]))
        return
    d0 = df_initial.loc[df_after_first.index[sec_mask]]
    d1 = df_after_first.loc[df_after_first.index[sec_mask]]
    d2 = df_final.loc[df_after_first.index[sec_mask]]

    display(HTML(f"<h4>{title_prefix} — {section_name}</h4>"))
    t = table_final_vs_after_first(
        d1, d2,
        metric='pre_DBH', vol_col='pre_stem_vol',
        status_col='status', thin_col='thin_decision',
        strategy=f'{strategy_dd.label}'
    ).round(3).set_index('Strategy')
    display(t)

def _on_change_stand(change):
    if change.get("name") != "value":
        return
    new_path = change["new"]

    # refresh globals used by render()
    global df, df_best3, best_start3, ranked3, df_k3_by_start

    df = load_stand_csv(new_path)
    df_best3, best_start3, ranked3 = compute_best3(df)
    df_k3_by_start = {s: k_row_thinning(df, 3, start_row=s, row_col="Row", status_col="status") for s in (1, 2, 3)}

    # Refresh the baseline dropdown to reflect the new "best"
    k3_start_dd.options = [(f"3-row start = {s}" + (" (best)" if s == best_start3 else ""), s) for s in (1, 2, 3)]
    k3_start_dd.value = best_start3  
    
stand_dd.observe(_on_change_stand, names="value")


def render():
    for box in (out_full, out_release, out_s1, out_s2, out_s3, out_s4, out_map):
        with box: clear_output(wait=True)

    df_after_first = df_k3_by_start[int(k3_start_dd.value)]

    key = strategy_dd.value
    params = dict(
        removed_frac = float(removed_frac_sl.value),
        anchor_frac  = float(anchor_frac_sl.value),
        k_neighbors  = int(k_neighbors_sl.value),
        row_scale    = float(row_scale_sl.value) if 'row_scale_sl' in globals() else 1.0,
        tree_scale   = float(tree_scale_sl.value) if 'tree_scale_sl' in globals() else 1.0,
    )

    # Run the sectional secondary strategy ON THE SELECTED BASELINE
    df_out, QB = run_sectional_strategy(df_after_first, key, **params)

    # 1) Whole-stand analysis
    with out_full:
        display(HTML("<h3>Analysis of Complete Stand</h3>"))
        whole_tbl = table_final_vs_after_first(
            df_after_first=df_after_first,
            df_final=df_out,
            metric='pre_DBH', vol_col='pre_stem_vol',
            status_col='status', thin_col='thin_decision',
            strategy=f'{strategy_dd.label}'
        ).round(3).set_index('Strategy')
        display(whole_tbl)

    # 2) Release table (whole stand)
    with out_release:
        display(HTML("<h3>Release Tables</h3>"))
        rel_tbl = anchor_release_table_immediate5(
            df_after_first=df_after_first,
            df_final=df_out,
            treatment=f'{strategy_dd.label}',
            top_pct_anchors=float(anchor_frac_sl.value),
            neighbors_k=int(k_neighbors_sl.value),
            dbh_col='pre_DBH', row_col='Row', tree_col='Tree',
            status_col='status', thin_col='thin_decision',
            keep_val='Keep', thin_val='Thin',
            row_scale=params['row_scale'], tree_scale=params['tree_scale']
        ).round(3).set_index('Treatment')
        display(rel_tbl)

    # 3–6) Section tables
    with out_s1:
        _show_section_table(df, df_after_first, df_out, QB['NW'], 'section-1', 'Analysis')
    with out_s2:
        _show_section_table(df, df_after_first, df_out, QB['NE'], 'section-2', 'Analysis')
    with out_s3:
        _show_section_table(df, df_after_first, df_out, QB['SW'], 'section-3', 'Analysis')
    with out_s4:
        _show_section_table(df, df_after_first, df_out, QB['SE'], 'section-4', 'Analysis')

    # Map medians
    with out_map:
        r_med = int(np.median(df_after_first['Row'].to_numpy()))
        x_med = int(np.median(df_after_first['Tree'].to_numpy()))
        plot_thinning_map_with_sections(
            df_out, row_col='Row', x_col='Tree', status_col='status',
            title=strategy_dd.label + ' — spatial map',
            r_med=r_med, x_med=x_med
        )
        plt.show()

for w in [k3_start_dd, strategy_dd, removed_frac_sl, anchor_frac_sl, k_neighbors_sl,
          *( [row_scale_sl, tree_scale_sl] if 'row_scale_sl' in globals() and 'tree_scale_sl' in globals() else [] )]:
    w.observe(lambda ch: render(), names='value')

display(HTML("<b>Sectional secondary thinning — Choose a baseline and strategy:</b>"))
display(ctrls, out_full, out_release, out_s1, out_s2, out_s3, out_s4, out_map)
render()

Output()

Output()

Output()

Output()

Output()

Output()

Output()